# Kaggle ComfyUI Generation Queue Service

One-click GPU generation service with watchdog, job queue, and ngrok tunnel.

## Prerequisites
- Phone-verified Kaggle account with GPU quota
- ngrok authtoken saved as Kaggle Secret `NGROK_AUTHTOKEN`
- ComfyUI auth password saved as Kaggle Secret `COMFYUI_AUTH_PASSWORD`

## Quick Start
Run All cells below. When Cell 5 prints your ngrok URL, open it in a new tab.

In [ ]:
# =============================================================
# Cell 2: Inline Setup — installs deps, clones ComfyUI,
# downloads models, creates directories (REPLACES scripts/setup.sh)
# Idempotent: skips if setup_complete.flag exists
# =============================================================
import os
import subprocess

FLAG_PATH = "/kaggle/working/setup_complete.flag"

if os.path.exists(FLAG_PATH):
    print("⏭️ Setup already complete (flag exists). Skipping.")
else:
    print("🔧 Starting inline setup...")

    # --- Step 1: System dependencies ---
    print("\n📦 Installing system packages...")
    subprocess.run([
        "apt-get", "update", "-qq"
    ], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "libgl1", "libglib2.0-0", "ffmpeg", "git", "wget", "curl"
    ], check=True)

    # --- Step 2: Clone ComfyUI at pinned commit ---
    COMFYUI_DIR = "/kaggle/working/ComfyUI"
    PINNED_COMMIT = "a1b5c8d"
    if not os.path.exists(COMFYUI_DIR):
        print("\n🔄 Cloning ComfyUI...")
        subprocess.run([
            "git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFYUI_DIR
        ], check=True)
        subprocess.run([
            "git", "-C", COMFYUI_DIR, "checkout", PINNED_COMMIT
        ], check=True)
    else:
        print(f"\n✅ ComfyUI already exists at {COMFYUI_DIR}")

    # --- Step 3: Python dependencies ---
    print("\n🐍 Installing Python dependencies...")
    subprocess.run([
        "pip", "install", "-q",
        "fastapi==0.115.4",
        "uvicorn==0.32.0",
        "aiosqlite==0.20.0",
        "pydantic==2.9.2"
    ], check=True)

    # Install ComfyUI requirements
    comfyui_reqs = os.path.join(COMFYUI_DIR, "requirements.txt")
    if os.path.exists(comfyui_reqs):
        subprocess.run([
            "pip", "install", "-q", "-r", comfyui_reqs
        ], check=True)

    # --- Step 4: Download models ---
    MODELS_DIR = "/kaggle/working/models"
    os.makedirs(f"{MODELS_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{MODELS_DIR}/clip", exist_ok=True)
    os.makedirs(f"{MODELS_DIR}/vae", exist_ok=True)
    os.makedirs(f"{MODELS_DIR}/unet", exist_ok=True)

    print("\n⬇️ Downloading models (this may take 10-15 minutes)...")

    # MiniMax H3 Video model
    subprocess.run([
        "wget", "--retry=3", "-c", "-q", "--show-progress",
        "-O", f"{MODELS_DIR}/checkpoints/minimax_h3.safetensors",
        "https://huggingface.co/MiniMax/MiniMax-H3/resolve/main/minimax_h3.safetensors"
    ], check=False)

    # Flux Dev model
    subprocess.run([
        "wget", "--retry=3", "-c", "-q", "--show-progress",
        "-O", f"{MODELS_DIR}/unet/flux_dev.safetensors",
        "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors"
    ], check=False)

    # SDXL Base model
    subprocess.run([
        "wget", "--retry=3", "-c", "-q", "--show-progress",
        "-O", f"{MODELS_DIR}/checkpoints/sdxl_base.safetensors",
        "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"
    ], check=False)

    # --- Step 5: Create directory structure ---
    print("\n📁 Creating directory structure...")
    dirs = [
        "/kaggle/working/output",
        "/kaggle/working/logs",
        "/kaggle/working/queue",
        "/kaggle/working/scripts",
        "/kaggle/working/configs",
        "/kaggle/working/workflows"
    ]
    for d in dirs:
        os.makedirs(d, exist_ok=True)

    # --- Step 6: Copy configs and workflows from repo ---
    print("\n📋 Copying configs and workflows...")
    REPO_DIR = "/kaggle/input/kaggle-comfyui-queue"
    if os.path.exists(os.path.join(REPO_DIR, "configs")):
        subprocess.run([
            "cp", "-r", f"{REPO_DIR}/configs/.", "/kaggle/working/configs/"
        ], check=False)
    if os.path.exists(os.path.join(REPO_DIR, "workflows")):
        subprocess.run([
            "cp", "-r", f"{REPO_DIR}/workflows/.", "/kaggle/working/workflows/"
        ], check=False)
    if os.path.exists(os.path.join(REPO_DIR, "scripts")):
        subprocess.run([
            "cp", "-r", f"{REPO_DIR}/scripts/.", "/kaggle/working/scripts/"
        ], check=False)

    # --- Step 7: Mark setup complete ---
    with open(FLAG_PATH, "w") as f:
        f.write("setup complete\n")

    print("\n✅ Setup complete! Proceed to Cell 3.")

In [ ]:
# =============================================================
# Cell 3: Launch ngrok tunnel
# =============================================================
import os, subprocess, json, time

auth_token = os.environ.get("NGROK_AUTHTOKEN")
password = os.environ.get("COMFYUI_AUTH_PASSWORD")

if not auth_token:
    raise RuntimeError("NGROK_AUTHTOKEN not set. Add it via Kaggle Secrets.")
if not password:
    raise RuntimeError("COMFYUI_AUTH_PASSWORD not set. Add it via Kaggle Secrets.")

# Configure ngrok
subprocess.run(["ngrok", "config", "add-authtoken", auth_token], check=True)

# Wait for ComfyUI port 8188 to be ready
print("⏳ Waiting for ComfyUI to start on port 8188...")
for i in range(30):
    try:
        import urllib.request
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("✅ ComfyUI is ready.")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI did not start within 60 seconds")

# Launch ngrok
proc = subprocess.Popen(
    ["ngrok", "http", "8188", f"--basic-auth=kaggle:{password}", "--log=stdout", "--log-format=json"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
for line in proc.stdout:
    try:
        data = json.loads(line)
        if "url" in data and data["url"].startswith("https://"):
            url = data["url"]
            break
    except json.JSONDecodeError:
        continue

if not url:
    raise RuntimeError("Failed to extract ngrok URL within 30 seconds")

with open("/kaggle/working/ngrok_url.txt", "w") as f:
    f.write(url)

print(f"✅ ngrok tunnel active: {url}")
print(f"   Username: kaggle")
print(f"   Password: [your COMFYUI_AUTH_PASSWORD secret]")

In [ ]:
# =============================================================
# Cell 4: Start Queue API + Consumer + Watchdog as background processes
# =============================================================
import subprocess, os, json

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/ComfyUI:" + env.get("PYTHONPATH", "")

# Start Queue API
api_proc = subprocess.Popen(
    ["python3", "/kaggle/working/scripts/queue_api.py"],
    cwd="/kaggle/working", env=env,
    stdout=open("/kaggle/working/api.log", "w"),
    stderr=subprocess.STDOUT
)

# Start Queue Consumer
consumer_proc = subprocess.Popen(
    ["python3", "/kaggle/working/scripts/queue_consumer.py"],
    cwd="/kaggle/working", env=env,
    stdout=open("/kaggle/working/consumer.log", "w"),
    stderr=subprocess.STDOUT
)

# Start Watchdog (monitors and restarts comfyui, ngrok, api, consumer)
watchdog_proc = subprocess.Popen(
    ["python3", "/kaggle/working/scripts/watchdog.py"],
    cwd="/kaggle/working", env=env,
    stdout=open("/kaggle/working/watchdog_stdout.log", "w"),
    stderr=subprocess.STDOUT
)

pids = {
    "api": api_proc.pid,
    "consumer": consumer_proc.pid,
    "watchdog": watchdog_proc.pid
}
with open("/kaggle/working/service_pids.json", "w") as f:
    json.dump(pids, f)

print(f"✅ Services started:")
print(f"   Queue API PID: {api_proc.pid}")
print(f"   Consumer PID: {consumer_proc.pid}")
print(f"   Watchdog PID: {watchdog_proc.pid}")
print(f"   Logs: /kaggle/working/{{api,consumer,watchdog_stdout}}.log")

In [ ]:
# =============================================================
# Cell 5: Print ngrok URL and health status
# =============================================================
import json, os

# Read ngrok URL
try:
    with open("/kaggle/working/ngrok_url.txt") as f:
        url = f.read().strip()
except FileNotFoundError:
    url = "NOT READY — check Cell 3 output"

# Read health
try:
    with open("/kaggle/working/watchdog_health.json") as f:
        health = json.load(f)
except FileNotFoundError:
    health = {"status": "initializing"}

print("=" * 60)
print("🚀 KAGGLE COMFYUI GENERATION SERVICE READY")
print("=" * 60)
print(f"\n🌐 Access URL: {url}")
print(f"   Auth: kaggle / [COMFYUI_AUTH_PASSWORD]")
print(f"\n📊 Health: {json.dumps(health, indent=2)}")
print(f"\n📋 Queue API: http://127.0.0.1:8189/api/health")
print(f"📁 Output dir: /kaggle/working/output/")
print(f"📄 Logs: /kaggle/working/{{watchdog,api,consumer}}.log")
print("=" * 60)

## Submitting Jobs

### Via curl (from your local machine):
```bash
curl -u kaggle:YOUR_PASSWORD -X POST https://YOUR_NGROK_URL/api/jobs \
  -H "Content-Type: application/json" \
  -d '{"workflow": "minimax_h3_video", "params": {"prompt": "A cat in space", "duration_seconds": 10}}'
```

### Check job status:
```bash
curl -u kaggle:YOUR_PASSWORD https://YOUR_NGROK_URL/api/jobs/1
```

### List pending jobs:
```bash
curl -u kaggle:YOUR_PASSWORD https://YOUR_NGROK_URL/api/jobs?status=pending
```

## Troubleshooting
- **ngrok URL not working**: Check Cell 3 output for errors. Verify NGROK_AUTHTOKEN secret.
- **ComfyUI crashed**: Check /kaggle/working/watchdog.log. Watchdog auto-restarts within 5s.
- **Job stuck in running**: Consumer heartbeat recovery resets stale jobs after 90s.
- **OOM errors**: Use Draft quality or shorter duration. VRAM safety protocol auto-defers oversized jobs.
- **Disk full**: Watchdog auto-rotates outputs when usage exceeds 80%.